# 23 - vLLM attention backends and tuned settings on the T4
Question: does the cold-prefill gap to vLLM come from vLLM's default settings or from its attention path? Three configurations, each with prefix caching off and `--max-model-len 8192`: **tuned** (default Triton attention, `--max-num-batched-tokens 16384`), **flex** (`FLEX_ATTENTION`, the other backend vLLM lists as valid on a T4) and **flashinfer** (`FLASHINFER`, forced; vLLM's own log says FlashInfer is unsupported on compute capability 7.5, so this one may be refused). For each configuration that starts and really uses the requested backend, the notebook runs the cold prefill sweep (2,048 / 4,096 / 8,000 tokens) and a short ShareGPT run (16 and 64 clients).

Run on a **GPU T4 x2** session with Internet on and Persistence = Variables and Files, then press **Run All** (about 30-60 minutes: the vLLM install is about 8 minutes and each server start 3-5 minutes; a failing configuration ends quickly with its log tail printed). Do not use *Factory reset*.

In [ ]:
import os, sys, re, json, time, signal, shutil, subprocess, threading, glob

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = "/kaggle/working"
MODEL = f"{WORK}/models/Qwen2.5-1.5B-Instruct"
SHAREGPT = f"{WORK}/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
REPO = f"{WORK}/turingserve"
RESULTS = f"{WORK}/results"
PORT = 8001
RUN_SHAREGPT = True      # also a short ShareGPT run (16 and 64 clients) on every configuration that starts
SKIP = ["tuned"]         # configurations already measured (tuned: no gain over the default); remove a name to run it again
# name, backend to force (None = vLLM's own choice), extra `vllm serve` flags
CONFIGS = [
    ("tuned", None, ["--max-num-batched-tokens", "16384"]),     # default attention backend, larger prefill batch
    ("flex", "FLEX_ATTENTION", []),                             # the other backend vLLM lists as valid on the T4
    ("flashinfer", "FLASHINFER", []),                           # the backend vLLM's issue #60821 reports at 1.1 s for 8k tokens
]
os.makedirs(RESULTS, exist_ok=True)
os.environ["TS_SHAREGPT"] = SHAREGPT
os.environ["TS_KAGGLE"] = "1"
srv = None


def gpu_used_mib():
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
                       capture_output=True, text=True)
    return int(r.stdout.strip().splitlines()[0]) if r.returncode == 0 and r.stdout.strip() else -1


def stop_server():
    global srv
    if srv is not None and srv.poll() is None:
        try:
            os.killpg(os.getpgid(srv.pid), signal.SIGTERM)
            srv.wait(timeout=30)
        except Exception:
            try:
                os.killpg(os.getpgid(srv.pid), signal.SIGKILL)
            except Exception:
                pass
    srv = None
    subprocess.run(["pkill", "-f", "vllm serve"], capture_output=True)
    time.sleep(3)
    for _ in range(30):
        if 0 <= gpu_used_mib() < 500:
            break
        time.sleep(2)


def start_vllm(name, backend, extra):
    """Launch vLLM with prefix caching off. A forced backend is requested through the environment variable first, then
    through the CLI flag. Returns (backend vLLM reports using, how it was requested, log path), or (None, None, log path)."""
    global srv
    import requests
    # vLLM 0.33 ignores the VLLM_ATTENTION_BACKEND environment variable ("Unknown vLLM environment variable"), so the CLI flag
    # goes first; the environment form stays as a fallback for other versions.
    forms = [("cli", {}, ["--attention-backend", backend])] if backend else []
    forms.append(("env", {"VLLM_ATTENTION_BACKEND": backend} if backend else {}, []))
    log_path = f"{WORK}/vllm_{name}.log"
    fallback = (None, None, log_path)
    for form, env_add, cli_add in forms:
        stop_server()
        srv = subprocess.Popen(["vllm", "serve", MODEL, "--served-model-name", "qwen", "--dtype", "half", "--max-model-len", "8192",
                                "--gpu-memory-utilization", "0.85", "--port", str(PORT), "--no-enable-prefix-caching",
                                *extra, *cli_add], env=dict(os.environ, **env_add), stdout=open(log_path, "w"),
                               stderr=subprocess.STDOUT, start_new_session=True)
        t0 = time.time()
        up = False
        for i in range(300):                       # up to 25 minutes (a first-time kernel build can be slow)
            try:
                if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
                    up = True
                    break
            except Exception:
                pass
            if srv.poll() is not None:
                break
            if i % 12 == 11:
                print(f"  [{name}/{form}] still starting... {time.time() - t0:.0f}s", flush=True)
            time.sleep(5)
        txt = open(log_path, errors="replace").read()
        if up:
            m = re.search(r"Using (\w+) attention backend", txt)
            actual = m.group(1) if m else None
            print(f"[{name}] server up after {time.time() - t0:.0f}s via {form}; attention backend reported: {actual or '?'}")
            if backend and actual != backend:
                print(f"[{name}] vLLM did not use {backend} via {form}; trying the next way to request it")
                fallback = (actual, form, log_path)
                stop_server()
                continue
            return actual, form, log_path
        print(f"[{name}] did not start via {form}; last log lines:")
        print(txt[-1500:])
        stop_server()
    return fallback


def run_bench(engine, label, extra):
    cmd = [sys.executable, "-u", "-m", "benchmarks.serve_bench", "--base-url", f"http://localhost:{PORT}", "--engine", engine,
           "--model", "qwen", "--tokenizer", MODEL, "--out", RESULTS, "--label", label, *extra]
    p = subprocess.Popen(cmd, cwd=REPO, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Token indices sequence length" in line or "Running this sequence" in line:
            continue
        print(line, end="", flush=True)
    p.wait()
    print(f"[{label}] exit code {p.returncode}\n")
    return p.returncode


print("setup ok; GPU 0 memory used:", gpu_used_mib(), "MiB")

## 1. Install vLLM

In [ ]:
def can_import(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not can_import("vllm"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError("pip install vllm failed")
# the pip warnings about numba, gradio, pillow and opentelemetry concern other preinstalled packages and are harmless
r = subprocess.run([sys.executable, "-c", "import vllm, torch; print(vllm.__version__, torch.__version__)"], capture_output=True, text=True)
print("vllm / torch:", r.stdout.strip() or r.stderr[-1500:])
open(f"{WORK}/vllm_versions.txt", "w").write(r.stdout)

## 2. Model, dataset and benchmark client at tag `m3-rc`

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

if not os.path.exists(f"{MODEL}/model.safetensors"):
    snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=MODEL)
if RUN_SHAREGPT and not os.path.exists(SHAREGPT):
    hf_hub_download("anon8231489123/ShareGPT_Vicuna_unfiltered", "ShareGPT_V3_unfiltered_cleaned_split.json",
                    repo_type="dataset", local_dir=f"{WORK}/datasets/sharegpt")
if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "https://github.com/Muskanpandey0304/turingserve.git", REPO], check=True)
subprocess.run(["git", "fetch", "--tags", "--force"], cwd=REPO, check=True, capture_output=True)
subprocess.run(["git", "checkout", "-f", "m3-rc"], cwd=REPO, check=True, capture_output=True)
print("repo at:", subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO, capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "datasets", "transformers"], capture_output=True)
print("model ok:", os.path.exists(f"{MODEL}/model.safetensors"), "| dataset ok:", os.path.exists(SHAREGPT))

## 3. Run every configuration
A refused or failing configuration prints its log tail and the notebook moves on; that outcome is itself the result.

In [ ]:
outcomes = []
for name, backend, extra in CONFIGS:
    if name in SKIP:
        print(f"configuration {name}: skipped (already measured)")
        continue
    print("=" * 70, f"\nconfiguration: {name}  (forced backend: {backend or 'none'}, extra flags: {extra})")
    note = ""
    if name == "flashinfer":
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flashinfer-python"], capture_output=True, text=True)
        note = "pip install flashinfer-python " + ("ok" if r.returncode == 0 else "failed: " + r.stderr[-300:].replace("\n", " "))
        print(note)
    actual, form, log_path = start_vllm(name, backend, extra)
    row = {"name": name, "requested": backend, "actual": actual, "via": form, "started": actual is not None or form is not None,
           "benchmarked": False, "note": note}
    if not row["started"]:
        row["note"] += " server did not start (see log)"
    elif backend and actual != backend:
        row["note"] += f" vLLM used {actual} instead of the requested {backend}; not benchmarked"
    else:
        engine = f"vllm-{name}"
        run_bench(engine, "t4-1.5b-prefill-cold", ["--dataset", "prefill", "--num-prompts", "500", "--mode", "concurrency",
                                                    "--concurrency", "1", "--repeat", "3", "--gpu-log"])
        if RUN_SHAREGPT:
            run_bench(engine, "t4-1.5b-c16-64", ["--dataset", "sharegpt", "--num-prompts", "300", "--mode", "concurrency",
                                                  "--concurrency", "16,64", "--repeat", "1", "--ignore-eos", "--gpu-log"])
        row["benchmarked"] = True
    outcomes.append(row)
    stop_server()
json.dump(outcomes, open(f"{RESULTS}/vllm_backends.json", "w"), indent=1)
for o in outcomes:
    print(o)

## 4. Summary and download
Download **`/kaggle/working/vllm_backends_bundle.zip`**, then stop the session.

In [ ]:
stop_server()
REFERENCE = {2048: 939, 4096: 3204, 8000: 11065}      # vLLM with default settings, measured earlier (ms)
print(f"{'configuration':14}{'backend used':18}{'2048 tok':>10}{'4096 tok':>10}{'8000 tok':>10}   (cold TTFT median, ms)")
print(f"{'default (ref.)':14}{'TRITON_ATTN':18}" + "".join(f"{REFERENCE[k]:>10}" for k in sorted(REFERENCE)))
for o in json.load(open(f"{RESULTS}/vllm_backends.json")):
    files = glob.glob(f"{RESULTS}/*_vllm-{o['name']}_prefill_t4-1.5b-prefill-cold.json")
    if not (o["benchmarked"] and files):
        print(f"{o['name']:14}{str(o['actual'] or 'did not run'):18}  {o['note'].strip()[:90]}")
        continue
    r = json.load(open(files[0]))["runs"][0]
    by = {}
    for q in r["requests"]:
        if q.get("first") and not q.get("error"):
            by.setdefault(q["prompt_tokens"], []).append((q["first"] - q["send"]) * 1000)
    med = {L: sorted(v)[len(v) // 2] for L, v in by.items()}
    print(f"{o['name']:14}{str(o['actual']):18}" + "".join(f"{med.get(k, float('nan')):>10.0f}" for k in sorted(REFERENCE)))
    if med.get(8000, 1e9) < 300:
        print("   WARNING: 8k tokens in under 300 ms is not possible cold; check that prefix caching is really off")
print()
for f in sorted(glob.glob(f"{RESULTS}/*_vllm-*_sharegpt_*.json")):
    d = json.load(open(f))
    for r in d["runs"]:
        m = r["median"]
        print(f"{os.path.basename(f)[:60]:62} c={r['level']:>3} {m['output_throughput_tok_s']:8.1f} tok/s  errors {m['errors']}")

bundle = f"{WORK}/vllm_backends_bundle"
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle)
for f in glob.glob(f"{RESULTS}/*vllm*") + glob.glob(f"{WORK}/vllm_*.log") + [f"{WORK}/vllm_versions.txt"]:
    if os.path.isfile(f):
        shutil.copy(f, bundle)
shutil.make_archive(bundle, "zip", bundle)
print("\nDownload:", bundle + ".zip", f"({os.path.getsize(bundle + '.zip') / 1e6:.1f} MB)")